# PyTorch Basics: Tensors, Device & Autograd

PyTorch is an open-source deep learning library. This notebook covers the **core building blocks**: tensors, GPU/CPU device management, and automatic differentiation (autograd).

In [1]:
# Install PyTorch (run once)
!pip install torch torchvision torchaudio

## 1. Imports & Reproducibility

In [2]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Reproducibility: same results every run
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
print("Seed set to:", seed)

Seed set to: 42


## 2. Device Selection (CPU / GPU)

In [3]:
# Automatically pick the best available hardware
if torch.cuda.is_available():
    device = torch.device("cuda")       # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = torch.device("mps")        # Apple Silicon GPU
else:
    device = torch.device("cpu")        # Fallback

print("Using device:", device)

Using device: cpu


## 3. Tensors
Tensors are like NumPy arrays but they can run on GPU and support autograd.

In [4]:
# Create a 2x2 tensor
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("Tensor:", x)
print("Shape:", x.shape)
print("Dtype:", x.dtype)

# Move to device
x = x.to(device)
print("Device:", x.device)

Tensor: tensor([[1., 2.],
        [3., 4.]])
Shape: torch.Size([2, 2])
Dtype: torch.float32
Device: cpu


In [5]:
# Other ways to create tensors
zeros = torch.zeros(3, 3)
ones  = torch.ones(2, 4)
rand  = torch.randn(3, 3)   # normal distribution

print("Zeros:\n", zeros)
print("Ones:\n", ones)
print("Random:\n", rand)

Zeros:
 tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])
Ones:
 tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.]])
Random:
 tensor([[ 0.3367,  0.1288,  0.2345],
        [ 0.2303, -1.1229, -0.1863],
        [ 2.2082, -0.6380,  0.4617]])


In [6]:
# Basic tensor operations
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

print("Add:", a + b)
print("Multiply:", a * b)
print("Dot product:", torch.dot(a, b))
print("Mean:", a.mean())
print("Sum:", a.sum())

Add: tensor([5., 7., 9.])
Multiply: tensor([ 4., 10., 18.])
Dot product: tensor(32.)
Mean: tensor(2.)
Sum: tensor(6.)


## 4. Autograd — Automatic Differentiation
PyTorch tracks operations on tensors and computes gradients automatically using `requires_grad=True`.

In [7]:
# requires_grad=True tells PyTorch to track this tensor for gradients
w = torch.randn(2, 1, device=device, requires_grad=True)
b_bias = torch.zeros(1, device=device, requires_grad=True)

x_in = torch.tensor([[1.0, 2.0], [3.0, 4.0]], device=device)

# Forward: linear operation y = Xw + b
y_hat = x_in @ w + b_bias        # matrix multiply + bias

# A dummy loss (mean of squared predictions)
loss = (y_hat ** 2).mean()

print("y_hat:", y_hat)
print("loss:", loss.item())

# Backward: compute gradients
loss.backward()

print("Gradient of w:", w.grad)
print("Gradient of b:", b_bias.grad)

y_hat: tensor([[1.3372],
        [2.9417]], grad_fn=<AddBackward0>)
loss: 5.2207136154174805
Gradient of w: tensor([[10.1622],
        [14.4410]])
Gradient of b: tensor([4.2788])


### What's happening above?
1. **Forward pass**: compute `y = Xw + b`, then `loss`  
2. **Backward pass**: `loss.backward()` computes ∂loss/∂w and ∂loss/∂b automatically  
3. **Gradients** tell us: "increase w → loss goes up by this much"  
   → optimizer uses these to **update w** in the right direction

## 5. Simple Gradient Descent — Manual Example

In [8]:
# Simple 1D example: learn w such that y = 2x
# True weight: w = 2.0
torch.manual_seed(42)

w = torch.randn(1, requires_grad=True)
lr = 0.1

x_data = torch.tensor([1.0, 2.0, 3.0, 4.0])
y_data = torch.tensor([2.0, 4.0, 6.0, 8.0])   # y = 2*x

for step in range(20):
    y_pred = w * x_data
    loss = ((y_pred - y_data) ** 2).mean()

    loss.backward()

    with torch.no_grad():          # don't track this update
        w -= lr * w.grad
    w.grad.zero_()                 # clear gradient for next step

    if step % 5 == 0:
        print(f"Step {step:02d} | w={w.item():.4f} | loss={loss.item():.4f}")

print(f"\nLearned w: {w.item():.4f}  (true w = 2.0)")

Step 00 | w=2.8317 | loss=20.7495
Step 05 | w=1.9740 | loss=0.0203
Step 10 | w=2.0008 | loss=0.0000
Step 15 | w=2.0000 | loss=0.0000

Learned w: 2.0000  (true w = 2.0)
